In [1]:
import pandas as pd
import numpy as np
import random

In [170]:
class MyDBSCAN:
    def __init__(self, eps = 3, min_samples=3, metric="euclidean"):
        self.eps = eps
        self.min_samples = min_samples
        self.metric = metric

    def __str__(self):
        return f"MyDBSCAN class: eps={self.eps}, min_samples={self.min_samples}"

    def distance(self, X, y):
        if self.metric == "euclidean":
            return ((X - y) ** 2).sum(axis=1) ** 0.5
        if self.metric == "chebyshev":
            return (X - y).abs().max(axis=1) 
        if self.metric == "manhattan":
            return (X - y).abs().sum(axis=1)
        if self.metric == "cosine":
            return  1 - (X * y).sum(axis=1) / (((X ** 2).sum(axis=1) ** 0.5) * ((y ** 2).sum() ** 0.5) )
    
    def fit_predict(self, X: pd.DataFrame):
        probably_outliers = set()
        borders = set()
        roots = set()
        clusters = []
        neighbors = dict()
        cluster_points = set()
        
        for index, row in X.iterrows():
            if index not in probably_outliers and index not in borders\
                and index not in roots:
                distances_df = self.distance(X, row)
    
                mask = distances_df < self.eps
            
                if mask.sum() < self.min_samples + 1:
                    probably_outliers.add(index)
                else:
                    roots.add(index)
                    clusters.append({index})
    
                    neighbors[index] = set(mask.index[mask].to_list())
                    neighbors[index].remove(index)
    
                    while neighbors[index]:
                        neighbor_index = neighbors[index].pop()

                        if neighbor_index not in cluster_points:
                            if neighbor_index in probably_outliers:
                                probably_outliers.remove(neighbor_index)
                                borders.add(neighbor_index)
                                clusters[-1].add(neighbor_index)
                                cluster_points.add(neighbor_index)
                                continue
                            
                            neighbor_distances_df = self.distance(X, X.loc[neighbor_index])
        
                            neighbor_mask = neighbor_distances_df < self.eps
        
                            if neighbor_mask.sum() < self.min_samples + 1:
                                borders.add(neighbor_index)
                                clusters[-1].add(neighbor_index)
                                cluster_points.add(neighbor_index)
                                continue
        
                            roots.add(neighbor_index)
                            
                            neighbors_of_neighbor = set(neighbor_mask.index[neighbor_mask].to_list())
                            clusters[-1].add(neighbor_index)
                            cluster_points.add(neighbor_index)
                            
                            neighbors_of_neighbor.difference_update(cluster_points)
        
                            neighbors[index].update(neighbors_of_neighbor)
        
        if probably_outliers:
            clusters.append(set(probably_outliers))


        y = []

        for cluster_index, cluster in enumerate(clusters):
            y.append(pd.Series(cluster_index, cluster))
        
        return pd.concat(y)
                    
                        

In [171]:
my_dbscan = MyDBSCAN(2, 4, "chebyshev")

In [172]:
my_dbscan.fit_predict(X)

64    0
1     0
2     0
6     0
10    0
     ..
56    3
57    3
58    3
62    3
63    3
Length: 70, dtype: int64

In [123]:
X = pd.DataFrame.from_dict({'col_0': {0: -12.376754611762045, 1: -7.42880691074505, 2: -5.976138177380236, 3: -1.8543583428011559, 4: -3.870154434365707, 5: -13.42898995137563, 6: -6.636933317781169, 7: -5.386681566608507, 8: 3.999880102471793, 9: -8.90086119838424, 10: -7.059652495102105, 11: 5.16203782373999, 12: -4.122863003609685, 13: -5.975537128532185, 14: -8.1944517798291, 15: -5.041275423888809, 16: 4.056093278288349, 17: -8.09802706397457, 18: 1.8740414012130309, 19: -2.542940684897585, 20: -0.27626035072033206, 21: -4.779257811855778, 22: 5.529286012204423, 23: 5.468037414737012, 24: 2.9475738354632046, 25: 5.450088759715089, 26: 0.9435738603095327, 27: 3.4419432416148767, 28: -4.013464153626242, 29: -9.818598139466404, 30: -3.6828835878901307, 31: 6.486045286216628, 32: 2.6720072204852343, 33: 1.1549242992511348, 34: -0.4540405779625454, 35: -5.9812707203484585, 36: 5.498924560149255, 37: -0.4528353427947778, 38: -10.593460491329392, 39: -6.821492204335332, 40: 0.06416700402358289, 41: 3.4894429693648514, 42: -8.204027700568867, 43: 1.319413713982025, 44: -0.6334590494510866, 45: -3.667741855083906, 46: 2.271428647583279, 47: -8.825544115029858, 48: 4.3507581302575, 49: -4.848312635165775, 50: -8.687752231783444, 51: -8.925107180899118, 52: 2.541835028171505, 53: -1.0700765423310292, 54: -6.662009520555842, 55: -1.9964078515388932, 56: -4.591121896896085, 57: -3.1848920792974798, 58: -6.6502252498125145, 59: -2.3403771113329404, 60: -9.241542035801032, 61: -4.010794347849762, 62: -13.635255794883614, 63: 2.590950071374499, 64: -6.969192288926149, 65: -9.319230168588817, 66: 7.217377642467978, 67: 3.4882819088286325, 68: -1.904291944137665, 69: -9.695114047952934}, 'col_1': {0: 6.271909608585306, 1: -5.987328164496581, 2: -3.0350181771110236, 3: 4.332858992935977, 4: 9.291592602472987, 5: -4.825353332337887, 6: -4.4584971169437235, 7: 9.953531174062501, 8: 1.8879829189340613, 9: -8.134502202237288, 10: -4.3712773485458865, 11: -2.926005625758707, 12: 7.7599674475783, 13: -8.492908979788757, 14: 7.137408126083285, 15: 9.799904459686505, 16: 1.084290764836021, 17: 7.976161095948426, 18: -1.017346876260945, 19: 6.37000880580857, 20: 8.036287568464758, 21: 5.483526874860093, 22: 0.6568238989402069, 23: 4.41203250174663, 24: 3.502360406042902, 25: 1.0104639828999407, 26: 0.17355916295396034, 27: 1.5090222422811892, 28: 13.644981589470666, 29: 3.6647355451684067, 30: 10.370686237163234, 31: 2.4015903869156583, 32: 6.1160087353641845, 33: 8.449845376981983, 34: 2.745707231913979, 35: 9.20335549721564, 36: -2.434430704466103, 37: 5.9621770032707655, 38: 6.504367549004282, 39: 7.60856730509589, 40: 3.35629776540672, 41: 9.637590620445856, 42: -5.596941010492556, 43: 2.465322773613541, 44: 5.345432632508863, 45: 7.8499617442726795, 46: 2.9027624206304106, 47: 6.737055082060836, 48: 1.2204104449675102, 49: -3.4895095218488894, 50: 13.481628196711918, 51: 4.4018278214498725, 52: 1.2001387443126954, 53: 0.8603100232102179, 54: -7.627627969440615, 55: 4.62303620225167, 56: -6.0582318191267355, 57: -8.175785138960066, 58: -11.849031879778178, 59: 5.45241566266468, 60: 8.33365005753505, 61: 8.28505175371513, 62: 7.257238226875661, 63: 7.429308441626979, 64: -2.968500453740931, 65: 8.077391271332734, 66: 4.301369981731229, 67: 4.615975249488021, 68: 4.231085516553827, 69: 5.317829742444656}})